# Tải DINOv3 ViT-L/16 → tạo Kaggle Dataset PRIVATE

1. Settings: **Internet ON** (không cần GPU).
2. Dán link file **`dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth`** từ email Meta vào `URL` (giữ trong dấu nháy).
3. Run All. Kiểm tra: dung lượng ~1.2 GB, model load được, in ra `patch tokens (1, 196, 1024)`.
4. Save Version → mở tab **Output** → **New Dataset** → đặt tên `dinov3-vitl16-lvd1689m` → **Private** → Create.

Không chia sẻ link hay dataset công khai (DINOv3 License). Link trong email hết hạn sau vài ngày.

In [ ]:
URL = "DAN_LINK_VAO_DAY"   # link của file dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth

In [ ]:
import os, hashlib, subprocess
from pathlib import Path
out = Path('/kaggle/working/dinov3'); out.mkdir(parents=True, exist_ok=True)
pth = out / 'dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'
if not pth.exists():
    subprocess.run(['wget', '-q', '-O', str(pth), URL], check=True)
print('size GB:', round(pth.stat().st_size / 1e9, 3))
sha = hashlib.sha256(pth.read_bytes()).hexdigest()
print('sha256:', sha, '| prefix 8aa4cbdd:', sha.startswith('8aa4cbdd'))
repo = out / 'dinov3_repo'
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/facebookresearch/dinov3', str(repo)], check=True)
print(sorted(os.listdir(out)))

In [ ]:
import sys, torch
sys.path.insert(0, str(repo))
from dinov3.hub.backbones import dinov3_vitl16
m = dinov3_vitl16(pretrained=False)
m.load_state_dict(torch.load(pth, map_location='cpu', weights_only=True), strict=True)
m.eval()
with torch.no_grad():
    f = m.forward_features(torch.randn(1, 3, 224, 224))
print({k: tuple(v.shape) for k, v in f.items() if hasattr(v, 'shape')})
print('patch tokens', tuple(f['x_norm_patchtokens'].shape), '| storage tokens', m.n_storage_tokens)
print('OK -> Save Version, rồi tạo Dataset PRIVATE từ Output')